# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [10]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.487028,0.381455,0.838235,0.886986
2,0.328676,0.357605,0.848039,0.893471
3,0.167169,0.416541,0.860294,0.901554


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.167169,0.416541,3,0.860294,0.901554


{'eval_loss': 0.4165409803390503,
 'eval_accuracy': 0.8602941176470589,
 'eval_f1': 0.9015544041450777}

## 4. Перевірка на власних прикладах

In [13]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 86.0%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 2.6%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.0%



## 0. Опрацюйте демо-ноутбук лекції 6

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука в Google Colab.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (відповіді на питання потрібно написати у демо-ноутбуці)

1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?
2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?
3. Яке пряме застосування базової моделі bert-base-uncased?
4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?
5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?
6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?
7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?
8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?
9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.
10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

### 0.2. Мої відповіді:

1. Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?

- Початковий імпорт виконувався 37 секунд.
- **Тренування (3 епохи)** - найдовша частина: 58 секунд на T4.
- Проблеми не виявлено.

2. Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?

`[CLS]` - це **спеціальний токен**, який BERT додає на початок кожної послідовності. Після проходження через усі шари трансформера його **вихідне представлення** акумулює інформацію з **усієї** послідовності завдяки механізму self-attention.

Його функція: для задач **класифікації послідовностей** (як-от MRPC - «чи є пара речень парафразом?») ми беремо **саме вектор `[CLS]`** і подаємо його на нову голову-класифікатор (у нашому випадку - `Linear(768, 2)`). Голова перетворює цей 768-вимірний вектор у 2 логити (ймовірності класів 0 і 1).

У ноутбуці це видно на кроці 2: `AutoModelForSequenceClassification` бере `bert.pooler` (який і витягує `[CLS]`) і додає `classifier`. Саме тому `[CLS]` - це «стиснутий зміст» усієї послідовності, придатний для класифікації.

3. Яке пряме застосування базової моделі bert-base-uncased?

Базова модель `bert-base-uncased` - це **екстрактор ознак**. Її пряме застосування:
- **Отримання ембедингів** для слів/речень (наприклад, для семантичного пошуку, кластеризації, рекомендацій).
- **База для донавчання** під конкретну задачу.
- **Заповнення пропусків** (`fill-mask`).

**Що вона НЕ вміє:** класифікувати, відповідати на питання, генерувати текст. Для цього потрібна **голова** або **інша архітектура**.

4. Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

**Так, варто.**

Зазвичай це наслідок **тренувальних даних**.

Способи боротьби:
1. **Дебіасування даних** - фільтрація й перебалансування тренувального корпусу.
2. **Counterfactual data augmentation** - заміна «man» на «woman» і навпаки, щоб модель не асоціювала професії зі статтю.
3. **Fine-tuning на дебіасованих даних** - донавчання на прикладах без стереотипів.
4. **Пост-обробка** - фільтрація на рівні інференсу.

Дану проблему повністю прибрати неможливо, бо мова й культура самі містять стереотипи. Тому важливо **мінімізувати** й **документувати** обмеження моделі.

5. Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?

Це **`token_type_ids`** (`segment_ids`). BERT приймає **два речення** як одну послідовність:

> [CLS] речення 1 [SEP] речення 2 [SEP]

`token_type_ids` позначає, **до якого речення належить кожен токен**:
- `0` — токени з **першого** речення (і `[CLS]`);
- `1` — токени з **другого** речення (і `[SEP]`).

У виводі ноутбука видно:

> ['[CLS]', 'revenue', ..., 'earlier', '.'] → [0, 0, 0, ..., 0, 0]
> [',', 'revenue', ..., 'earlier', '.', '[SEP]'] → [1, 1, 1, ..., 1, 1]


**Навіщо:** модель має розрізняти, яке речення де, щоб правильно порівнювати їх. Без цього BERT не знав би, де межа між реченнями.

6. Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?

У виводі є рядок:

> Linear(in_features=768, out_features=2, bias=True)

Це **нова голова-класифікатор** (`model.classifier`):
- **`in_features=768`** - розмірність вихідного вектору з кістяка BERT (embedding `[CLS]` має 768 вимірів у `bert-base-uncased`).
- **`out_features=2`** - кількість **класів** у нашій задачі: 0 (не парафраз) і 1 (парафраз).
- **`bias=True`** - додатковий зсув (як у будь-якому `Linear`).

Голова - це **один лінійний шар** `768 → 2`, який перетворює «стиснутий зміст» послідовності (вектор `[CLS]`) у 2 логити (ймовірності класів).

7. Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

**Це добре**.

У виводі:
- Кістяк (`model.bert`): **109 482 240** параметрів.
- Голова (`model.classifier`): **1 538** параметрів.

**Чому добре:**
1. Кістяк **уже навчений** на величезному корпусі - він «знає» мову (граматику, семантику). Ми не вчимо його з нуля, лише **адаптуємо** під нашу задачу з малим learning rate (2e-5).
2. Голова **маленька** - її легко навчити навіть на 3668 прикладах MRPC. Якби голова була великою, вона б перенавчилася (overfitting) на малих даних.

8. Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?

Для прикладу можемо взяти `compute_metrics`.

- **Accuracy** - частка **правильних** передбачень: `(TP + TN) / (TP + TN + FP + FN)`. У нашому випадку: скільки пар речень модель правильно класифікувала як "парафраз" або "не парафраз".
- **F1** - гармонійне середнє між **precision** і **recall** для класу "парафраз" (label=1): `2 * (P * R) / (P + R)`, де `P = TP / (TP + FP)`, `R = TP / (TP + FN)`.

9. Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.

In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=10,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.077079,0.733088,0.835784,0.880143
2,0.062006,0.789256,0.843137,0.888889
3,0.094774,0.982849,0.818627,0.864964
4,0.089116,0.901912,0.840686,0.887348
5,0.041172,0.933196,0.848039,0.891608
6,0.033728,1.108683,0.828431,0.878893
7,0.011114,1.116973,0.840686,0.889643
8,0.003150,1.090855,0.835784,0.884682
9,0.001140,1.098169,0.838235,0.886207
10,0.000232,1.104458,0.838235,0.886207


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000232,1.104458,10,0.838235,0.886207


{'eval_loss': 1.1044580936431885,
 'eval_accuracy': 0.8382352941176471,
 'eval_f1': 0.8862068965517241}

Попередні результати:

| Епоха | Training Loss | Validation Loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.487 | 0.381 | 0.838 | 0.887 |
| 2 | 0.329 | 0.358 | 0.848 | 0.893 |
| 3 | 0.167 | 0.417 | 0.860 | 0.902 |

Нові результати:

| Епоха | Training Loss | Validation Loss | Accuracy | F1 |
|---|---|---|---|---|
| 1 | 0.077 | 0.733 | 0.836 | 0.880 |
| 2 | 0.062 | 0.789 | 0.843 | 0.889 |
| 3 | 0.095 | 0.983 | 0.819 | 0.865 |
| 4 | 0.089 | 0.902 | 0.841 | 0.887 |
| 5 | 0.041 | 0.933 | 0.848 | 0.892 |
| 6 | 0.034 | 1.109 | 0.828 | 0.879 |
| 7 | 0.011 | 1.117 | 0.841 | 0.890 |
| 8 | 0.003 | 1.091 | 0.836 | 0.885 |
| 9 | 0.001 | 1.098 | 0.838 | 0.886 |
| 10 | 0.000 | 1.104 | 0.838 | 0.886 |

**В результаті:**
- **Training Loss** у новому запуску падає майже до нуля (0.077 → 0.000) — модель **запам'ятовує** тренувальні дані.
- **Validation Loss** зростає з кожною епохою (0.733 → 1.104): модель втрачає здатність узагальнювати.
- **Accuracy/F1** коливаються в межах 0.82–0.85, але **не покращуються** після 2-ї епохи.

**Чому так:** Це класичне **перенавчання (overfitting)**. MRPC - маленький датасет (3668 тренувальних прикладів). Після 3-ї епохи модель «вивчає напам'ять» тренувальні пари (loss падає до нуля), але на нових даних помиляється частіше (validation loss зростає). У реальних задачах це вирішують **ранньою зупинкою (early stopping)** - беруть модель з **мінімальним validation loss**, а не останню.

10. Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.

In [19]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    ("The cat sat on the mat.", "A feline rested on the rug."),
    ("I love programming in Python.", "Python is my favorite programming language."),
    ("The Eiffel Tower is in Paris.", "The Eiffel Tower is in London."),
    ("She is a doctor.", "He is a doctor."),
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 99.9%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 0.0%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 100.0%

The cat sat on the mat.
A feline rested on the rug.
→ парафраз: 99.7%

I love programming in Python.
Python is my favorite programming language.
→ парафраз: 100.0%

The Eiffel Tower is in Paris.
The Eiffel Tower is in London.
→ парафраз: 100.0%

She is a doctor.
He is a doctor.
→ парафраз: 0.6%



| № | Речення 1 | Речення 2 | Результат | Правильно? |
|---|---|---|---|---|
| 1 | «The cat sat on the mat.» | «A feline rested on the rug.» | парафраз: **99.7%** | ✅ так |
| 2 | «I love programming in Python.» | «Python is my favorite programming language.» | парафраз: **100.0%** | ✅ так |
| 3 | «The Eiffel Tower is in Paris.» | «The Eiffel Tower is in London.» | парафраз: **100.0%** | ❌ **помилка** |
| 4 | «She is a doctor.» | «He is a doctor.» | парафраз: **0.6%** | ❌ **помилка** |

**Аналіз помилок:**

- **№1, №2 - успіх.** Модель добре розпізнає синоніми («cat»/«feline», «love»/«favorite») і перефразування. Це очікувано: MRPC містить багато таких прикладів.

- **№3 - провал, але показовий.** «Ейфелева вежа в Парижі» vs «Ейфелева вежа в Лондоні» **фактично протилежні** твердження. Модель каже **100% парафраз**. Причина: MRPC - це новинний датасет, де структурно схожі речення (однакові слова, крім одного) часто є парафразами. Модель навчилася **структурній схожості**, але **не розуміє фактів**. Це обмеження BERT: він не «знає», що Лондон ≠ Париж.

- **№4 - провал в інший бік.** «She is a doctor» vs «He is a doctor» зміст **той самий** (професія), лише стать різна. Модель каже **0.6%** (категорично «ні»). Причина: у тренувальних даних MRPC пари з різними займенниками часто означають **різних людей** (не парафраз). Модель «перестрахувалася» й вирішила, що різна стать = різний зміст.

**Висновок:** BERT добре працює на **лексичних** парафразах (синоніми), але **провалюється** на:
- **Фактичних** перевірках.
- **Семантичних** нюансах.

Це показує, що **класичний BERT** обмежений: він добре розуміє **мову**, але не **світ**. Для задач, де потрібні факти, краще використовувати **більші LLM** (як Qwen, GPT) або **RAG**.